# PharmaVigil: MedDRA coder fine-tuning (Kaggle GPU)

Fine-tunes BiomedBERT on VAERS narratives → human-coded MedDRA PTs (ADR-007), scores micro/macro F1 and
top-5 recall on the latest year, exports `coder.onnx` with a parity check, and packages the evidence.

*Labels come only from VAERS's human coders. Not for clinical or regulatory use.*

**Settings:** Accelerator **GPU T4 x2**, Internet **on**. Input dataset: `pharmavigil-vaers` (the yearly
`<year>VAERSData.zip` files).

In [ ]:
GIT_REF = "main"  # use the pull-request branch name until it is merged
TEST_YEAR = 2025     # latest complete year in the uploaded files
SCOPE = f"VAERS domestic narratives, receipt years up to {TEST_YEAR}"
REPO = "https://github.com/sumitparmar19/PharmaVigil.git"


In [ ]:
!pip install -q "pharmavigil-data @ git+{REPO}@{GIT_REF}#subdirectory=data" "pharmavigil-ml[train] @ git+{REPO}@{GIT_REF}#subdirectory=ml"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
os.environ["PV_LAKE"] = "/kaggle/working/lake"
!pv-data vaers --zip /kaggle/input/pharmavigil-vaers/*.zip
!python -c "import pharmavigil_data.cli as c; print(open(c.ANALYSIS_DIR / 'vaers_run.json').read())"

## Fine-tune, evaluate, export (seed 13)

In [ ]:
!pv-ml coder --narratives /kaggle/working/lake/bronze/vaers/narratives.parquet --out /kaggle/working/m5 \
    --models-dir /kaggle/working/models --scope "{SCOPE}" --test-year {TEST_YEAR} --epochs 3 --batch-size 32

In [ ]:
!sha256sum /kaggle/working/models/coder/coder.onnx
!cd /kaggle/working && zip -qr pharmavigil_m5_coder.zip m5 models/coder/labels.json models/coder/coder_config.json models/coder/coder_vectors.json && ls -la pharmavigil_m5_coder.zip